# Transformer-based Training: Fine-tuning a Pre-trained Transformer

This tutorial trains the same five-component UD pipeline as the [main tutorial](tutorial.ipynb), but with a **pre-trained transformer** as the embedding backbone instead of a CNN. Any encoder-style transformer that `spacy-transformers` can load from Hugging Face can be the backbone — you choose the one that matches your language and domain (see the model table in Step 2).

As the worked example, this notebook fine-tunes the bundled recipe's default transformer — **MacBERTh** ([emanjavacas/MacBERTh](https://huggingface.co/emanjavacas/MacBERTh)), a BERT model pre-trained on historical English — on the same Winter's Tale corpus as the main tutorial, so the final scores compare directly against the CNN model. Everything in the workflow applies unchanged to any other transformer.

## Transformer vs CNN — which should you use?

| | Transformer (this tutorial) | CNN (main tutorial) |
| --- | --- | --- |
| Accuracy | Higher — builds on what the transformer already learned in pre-training | Lower on unfamiliar spelling/vocabulary — relies on your training data alone |
| Training hardware | NVIDIA GPU strongly recommended | CPU is fine |
| Model size | ~500 MB | ~20 MB |
| Inference speed | Slower | Fast |
| Deployment | Requires spacy-transformers + torch installed | Plain spaCy |

**Rule of thumb:** if a pre-trained transformer exists for your language and domain and you have a GPU, the transformer is usually worth it. If you need small, fast, CPU-friendly models, stick with the CNN.

## Before you start

- **Complete the [main tutorial](tutorial.ipynb) first.** This notebook reuses its data and compares scores against its trained model.
- **You need an NVIDIA GPU.** Transformer training on CPU is impractically slow (hours become days).
- **First run downloads the transformer weights** (~450 MB for the example model) from Hugging Face; they are cached afterwards.

## Prerequisites

Install the transformer extras (in addition to the base package). From the repository root:

```bash
pip install -e .[gpu,transformers]
```

This installs `spacy-transformers`, which pulls in `torch` and Hugging Face `transformers` at tested versions, plus the CUDA libraries from the `gpu` extra.

> **Windows GPU note:** the default `torch` wheel from PyPI on Windows is **CPU-only**. After installing, run the GPU check cell below. If `torch.cuda.is_available()` prints `False` but you have an NVIDIA GPU, reinstall torch from the PyTorch CUDA index (match the CUDA level to your driver):
>
> ```bash
> pip install torch --index-url https://download.pytorch.org/whl/cu126 --force-reinstall
> ```

In [ ]:
from pathlib import Path

from lexos.language_model import LanguageModel, split_conllu

## Check your GPU

Run the cell below before anything else. It verifies two separate things:

1. **`_has_nvidia_gpu()`** — an NVIDIA GPU and driver are visible to the system (this is what the module checks when you pass `gpu=True`)
2. **`torch.cuda.is_available()`** — your installed torch build can actually use it (catches the Windows CPU-only wheel problem)

Both must be `True` before you start training.

In [ ]:
import torch

from lexos.language_model import _has_nvidia_gpu

print(f"NVIDIA GPU detected:      {_has_nvidia_gpu()}")
print(f"torch can use CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:               {torch.cuda.get_device_name(0)}")
else:
    print()
    print("torch cannot see a GPU. If you have an NVIDIA GPU, your torch build")
    print("is probably CPU-only. Reinstall from the PyTorch CUDA index:")
    print("  pip install torch --index-url https://download.pytorch.org/whl/cu126 --force-reinstall")

## Configuration

**`_tutorial_dir`** — same convention as the main tutorial: derived from the installed package location, or replace with a direct path if you are working outside the Lexos repo.

**Project name (`winter_tale_trf`)** — this notebook writes all output into a `winter_tale_trf` subfolder, deliberately separate from the main tutorial's `winter_tale` so the CNN model stays intact for the score comparison at the end. Rename both as needed for your own data.

In [ ]:
import lexos.language_model as _lm

# Derive the tutorial folder from the installed package path.
# Works when running from the Lexos repo with pip install -e .
#   __init__.py → language_model/ → lexos/ → src/ → repo root
_repo_root = Path(_lm.__file__).parents[3]
_tutorial_dir = _repo_root / "src" / "docs" / "tutorials" / "language_model"

# Not in the Lexos repo? Replace the two lines above with a direct path:
# _tutorial_dir = Path("C:/Users/you/my_project")

print(f"Tutorial directory: {_tutorial_dir}")
if not _tutorial_dir.exists():
    print("WARNING: Directory not found — set _tutorial_dir above.")

---

## Step 1: Split your data

Same as the main tutorial: split one CONLL-U file into train/dev/test. The example reuses the Winter's Tale corpus from the main tutorial.

**If you already have separate train/dev/test files**, skip this step and pass them directly to `copy_assets()` in Step 3.

> **Keep your text in the form the transformer was pre-trained on.** A transformer reads raw text through its own subword tokenizer, so preprocessing that moves your text away from the model's pre-training data usually hurts accuracy. In the example here: MacBERTh was pre-trained on raw historical spelling (*haue*, *vnto*, *ſ* and all), so the Winter's Tale text must **not** be modernized. See [Tuning Training Settings](../../user_guide/language_model/training_settings.md) for the full normalization discussion.

In [ ]:
splits = split_conllu(
    input_path=str(_tutorial_dir / "wt_sanitized.conllu"),               # ← your CONLL-U data file
    output_dir=str(_tutorial_dir / "winter_tale_trf" / "assets" / "en"), # ← rename "winter_tale_trf"
    train_ratio=0.8,
    dev_ratio=0.1,
    seed=42,
)

for name, path in splits.items():
    print(f"{name:5s}: {path}")

---

## Step 2: Create the model from the transformer recipe

Transformer training uses a **recipe** — a complete, vetted config file — instead of the `base_model` dict. Pass `recipe="transformer_ud.cfg"` and the bundled recipe is loaded as-is; `base_model` sourcing is specific to CNN (tok2vec) pipelines and is not used here.

**Choosing your transformer.** Pick the Hugging Face model that matches your language and domain:

| Hugging Face model | Best for |
| --- | --- |
| `bert-base-uncased` | Modern English, general |
| `roberta-base` | Modern English, stronger but larger |
| `emanjavacas/MacBERTh` | Historical English (the recipe default, used as this notebook's example) |
| Any encoder-style transformer that spacy-transformers can load from Hugging Face | Non-English — also change `lang=` |

The recipe defaults to MacBERTh. To use any other transformer, edit the model name after creating the `LanguageModel`:

```python
model.config["components"]["transformer"]["model"]["name"] = "bert-base-uncased"
model.save_config()
```

In [ ]:
model = LanguageModel(
    model_dir=str(_tutorial_dir / "winter_tale_trf"),  # ← rename to your project name
    lang="en",       # ← your language BCP-47 code
    gpu=True,        # ← transformer training needs a GPU (see the check above)
    recipe="transformer_ud.cfg",
    force=True,      # regenerate config from scratch; False reconnects to an existing run
)

### What the recipe sets (and why it differs from the CNN config)

- **Learning rate: `warmup_linear.v1`, peak 5e-5** — the transformer arrives pre-trained; a learning rate as high as the CNN's 0.001 would overwrite (catastrophically forget) what it already knows. The rate warms up from zero over 250 steps, then decays. There is **one** learning rate for the whole network — spaCy has no separate per-component rate.
- **`accumulate_gradient = 3`** — gradients from 3 batches are accumulated before each weight update, simulating a 3× larger batch for stabler transformer updates without 3× the GPU memory.
- **Batcher: `batch_by_padded`, size 2000** — transformers process padded token blocks; this batcher controls the padded total per batch (the main GPU-memory knob).
- **Early stopping** — same mechanism as the CNN: `eval_frequency = 200`, `patience = 1600` (in *steps*), best checkpoint picked by weighted dev score.

For what these settings mean and how to tune them, see [Tuning Training Settings](../../user_guide/language_model/training_settings.md).

---

## Step 3: Copy and convert your data

In [ ]:
# The ** unpacks the dict from split_conllu() — equivalent to passing
# train=..., dev=..., test=... as keyword arguments.
model.copy_assets(**splits)

In [ ]:
# Convert CONLL-U files to spaCy's binary format.
# n_sents=1 keeps one sentence per training doc. spaCy's preflight check
# counts docs (not sentences) and refuses to train a pipeline with no
# sourced components on fewer than 100 docs. Grouping sentences (like the
# CNN tutorial's n_sents=10) makes a small corpus look 10x smaller and
# trips that check — the recipe's components are all factory-defined,
# so the check applies here even though the transformer is pre-trained.
model.convert_assets(n_sents=1)

---

## Step 4: Train

**First run:** training initialization downloads the transformer weights (~450 MB for the example model) from Hugging Face. This needs a network connection and takes a few minutes; the weights are cached (in `~/.cache/huggingface`) so later runs skip the download.

**GPU memory (VRAM) guidance:**

| VRAM | Settings |
| --- | --- |
| 8 GB+ | Recipe defaults are fine |
| 6 GB | `model.config["training"]["batcher"]["size"] = 1000`; `model.config["training"]["accumulate_gradient"] = 6`; `model.save_config()` |
| 4 GB | Batcher size 500, and set `model.config["components"]["transformer"]["model"]["mixed_precision"] = True`; `model.save_config()` |

If training crashes with a CUDA out-of-memory error, drop the batcher size and raise `accumulate_gradient` to compensate (their product ≈ constant effective batch).

**Smoke test first.** The cell below has a `SMOKE_TEST` flag. Leave it `True` for your first run — it caps training at 50 steps (~a few minutes including the download) to prove the whole pipeline works before you commit to a full run. Then set it to `False`, re-run from Step 2.

In [ ]:
SMOKE_TEST = False  # ← set to False for the real training run

if SMOKE_TEST:
    model.config["training"]["max_steps"] = 50
    model.config["training"]["eval_frequency"] = 25
    model.save_config()
    print("Smoke test: max_steps=50. Set SMOKE_TEST = False for a full run.")
else:
    print("Full run: recipe defaults (max_steps=20000, early stopping).")

In [ ]:
model.train()

---

## Step 5: Evaluate

Evaluate against the held-out test split, exactly as in the main tutorial. (Skip drawing conclusions from a smoke-test run — 50 steps is not a trained model.)

In [ ]:
model.evaluate()

In [ ]:
# Compare transformer scores against the CNN baseline from the main tutorial.
import json

_metrics = {
    "CNN (winter_tale)": _tutorial_dir / "winter_tale" / "metrics" / "en" / "en.json",
    "Transformer (winter_tale_trf)": _tutorial_dir / "winter_tale_trf" / "metrics" / "en" / "en.json",
}
_keys = ["tag_acc", "pos_acc", "morph_acc", "lemma_acc", "dep_uas", "dep_las"]
_labels = ["TAG", "POS", "MORPH", "LEMMA", "UAS", "LAS"]

_loaded = {}
for name, path in _metrics.items():
    if path.exists():
        _loaded[name] = json.loads(path.read_text(encoding="utf-8"))
    else:
        print(f"Missing: {path}")

if len(_loaded) == 2:
    (cnn_name, cnn), (trf_name, trf) = _loaded.items()
    print(f"{'Metric':8s}  {'CNN':>8s}  {'Transformer':>12s}  {'Delta':>8s}")
    print("-" * 44)
    for key, label in zip(_keys, _labels):
        c = cnn.get(key, 0.0) * 100
        t = trf.get(key, 0.0) * 100
        print(f"{label:8s}  {c:7.2f}%  {t:11.2f}%  {t - c:+7.2f}%")
else:
    print("Run the main tutorial (CNN) and this notebook (transformer) fully")
    print("to produce both metrics files, then re-run this cell.")

**Reading the comparison:** expect the transformer's biggest gains where its pre-training overlaps your data. In this notebook's example — MacBERTh on an Early Modern corpus — that means POS/MORPH and dependency scores, where exposure to historical spelling pays off. LEMMA may stay weak for both until you have more training data (the trainable lemmatizer needs examples regardless of backbone).

**Choosing for real work:**

- **Transformer** — accuracy-critical annotation; a pre-trained model matching your language/domain exists; GPU available for training; model size and inference speed are secondary.
- **CNN** — CPU-only environments, fast batch processing, small distributable models. A packaged transformer model is ~500 MB and requires `spacy-transformers` to be installed wherever it is loaded.

---

## Step 6: Use your model

In [ ]:
import spacy

# ← rename "winter_tale_trf" to your project name
nlp = spacy.load(str(_tutorial_dir / "winter_tale_trf" / "training" / "en" / "model-best"))

# Test text in the same form the transformer was pre-trained on —
# here, original Early Modern spelling with no modernization.
doc = nlp("If you shall chance, Camillo, to visit Bohemia on the like occasion whereon my services are now on foot.")

for token in doc:
    print(f"{token.text:20s}  POS: {token.pos_:8s}  TAG: {token.tag_:6s}  LEMMA: {token.lemma_}")

---

## Troubleshooting

**`✘ Low number of examples to train a new pipeline (N)` fails validation:** spaCy counts training *docs*, not sentences, and requires at least 100 docs when no component is sourced from an existing model (which is the case for all transformer recipes — spaCy cannot see that the transformer backbone is pre-trained). `convert_assets(n_sents=...)` groups that many sentences into each doc, shrinking the doc count. Re-run `model.convert_assets(n_sents=1)` so docs equal sentences. If you have fewer than 100 sentences total, you genuinely need more training data.

**CUDA out of memory:** lower `training.batcher.size` (2000 → 1000 → 500) and raise `training.accumulate_gradient` to compensate; enable `mixed_precision = True` on the transformer component as a last resort. Save the config and re-run training.

**`torch.cuda.is_available()` is False but you have an NVIDIA GPU:** you have the CPU-only Windows wheel. Reinstall: `pip install torch --index-url https://download.pytorch.org/whl/cu126 --force-reinstall` (use cu121/cu124 for older drivers).

**`LexosException` about spacy-transformers:** the transformer extras are not installed in this environment — `pip install -e .[transformers]` (or `.[gpu,transformers]`).

**Hugging Face download fails:** downloading the transformer weights needs a network connection (and no blocking proxy). The download is retried on the next `train()` call; once cached it never re-downloads.

**Reconnecting after a kernel restart:** run the imports cell, the configuration cell, and the `LanguageModel(...)` cell with `force=False` to reconnect to your existing `winter_tale_trf/` directory.

**Training is extremely slow:** confirm the GPU is actually engaged — run `nvidia-smi` in a terminal during training and check the python process appears with GPU memory allocated. If not, re-run the GPU check cell at the top.

**Loading the packaged model elsewhere fails:** environments that load a transformer-based model need `spacy-transformers` installed too, not just spaCy.

For training-setting tuning (learning rate, early stopping, batching, normalization), see [Tuning Training Settings](../../user_guide/language_model/training_settings.md).